In [0]:
catalog = 'olist'
base = f'/Volumes/{catalog}/bronze/landing'

In [0]:
# delete the broken table and Auto Loader's memory of it
spark.sql(f'DROP TABLE IF EXISTS {catalog}.bronze.reviews')
dbutils.fs.rm(f'{base}/_chk/reviews', True)
dbutils.fs.rm(f'{base}/_schema/reviews', True)

In [0]:
from pyspark.sql import functions as F
(spark.readStream.format('cloudFiles')
  .option('cloudFiles.format', 'csv')
  .option('header', 'true')
  .option('multiLine', 'true')
  .option('quote', '"')
  .option('escape', '"')
  .option('cloudFiles.schemaLocation', f'{base}/_schema/reviews')
  .load(f'{base}/reviews/')
  .withColumn('_ingested_at', F.current_timestamp())
  .withColumn('_source_file', F.col('_metadata.file_path'))
  .writeStream
  .option('checkpointLocation', f'{base}/_chk/reviews')
  .trigger(availableNow=True)
  .toTable(f'{catalog}.bronze.reviews'))